In [0]:
from pyspark.sql.functions import col, sum as _sum, to_date

df = spark.table("silver_transactions")

df = df.withColumn("date", to_date("timestamp"))
df = df.withColumn("revenue", col("price") * col("quantity"))

# Daily Revenue
daily_revenue = (
    df.groupBy("date")
      .agg(_sum("revenue").alias("total_revenue"))
)

# Product Performance
product_performance = (
    df.groupBy("product_id")
      .agg(
          _sum("revenue").alias("total_revenue"),
          _sum("quantity").alias("total_units")
      )
)

# Store Revenue
store_revenue = (
    df.groupBy("store_id")
      .agg(_sum("revenue").alias("total_revenue"))
)

# Payment Breakdown
payment_breakdown = (
    df.groupBy("payment_method")
      .agg(_sum("revenue").alias("total_revenue"))
)

# Write tables (simple + reliable)
daily_revenue.write.format("delta").mode("overwrite").saveAsTable("gold_daily_revenue")
product_performance.write.format("delta").mode("overwrite").saveAsTable("gold_product_performance")
store_revenue.write.format("delta").mode("overwrite").saveAsTable("gold_store_revenue")
payment_breakdown.write.format("delta").mode("overwrite").saveAsTable("gold_payment_breakdown")